# 第 28 章　分析の比較と選択

指定した配合と材齢について，試す前に強度を見積もります。直線と森を比べるとき，収録された一行を等しく数えるか，配合を一つずつ等しく数えるかで，平均の意味が変わります。講義では，検証行の予測をそろえ，どの方法で平均の誤差が小さく，どの条件で外れが残るかを読みます。

実習では第 27 章で保存した問いと基本の出力を使います。このノートを使って比較を確かめ，選んだ方法や集計と，次回までに指定する確認を自分のノートに残します。手元の出力がなければ，後半の Concrete の予測か記述から始められます。

Colab では「ドライブにコピーを保存」で自分用のノートを作ります。ローカルでは配布 ZIP の配置のまま開くか，ノートと CSV を同じフォルダーに置きます。ファイルが見つからなければ公開 URL から読み込みます。

## 講義　同じ予測を二通りに平均する

### 同じ検証行で比べる

学習 631 行，検証 208 行を使います。モデルへ渡すのは材料量七列と材齢だけです。同じ配合が学習と検証へまたがらない分割を使います。実測と予測を一行ずつ対応させて，そのまま検証行の表に残します。

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression

file_name = "concrete-strength.csv"
csv_path = Path("data") / file_name
if not csv_path.is_file():
    csv_path = Path("../data") / file_name
if not csv_path.is_file():
    csv_path = Path(file_name)
if not csv_path.is_file():
    csv_path = "https://raw.githubusercontent.com/sohei-takagi/python-ds-course-materials/main/data/" + file_name

concrete = pd.read_csv(csv_path)
train = concrete.loc[concrete["split"] == "train"]
valid = concrete.loc[concrete["split"] == "validation"]

features = [
    "cement", "slag", "fly_ash", "water", "superplasticizer",
    "coarse_aggregate", "fine_aggregate", "age",
]
X_train, y_train = train[features], train["strength"]
X_valid = valid[features]
print(f"Training: {len(train)} records, {train['recipe_id'].nunique()} recipes")
print(f"Validation: {len(valid)} records, {valid['recipe_id'].nunique()} recipes")

In [ ]:
line = LinearRegression().fit(X_train, y_train)
forest = RandomForestRegressor(
    n_estimators=200, max_features=0.7, random_state=2026, n_jobs=2,
)
forest.fit(X_train, y_train)

scored = valid[["record_id", "recipe_id", "age", "strength"]].copy()
scored["baseline"] = y_train.mean()
scored["linear"] = line.predict(X_valid)
scored["forest"] = forest.predict(X_valid)

原行 85 と 108 は同じ配合の 3 日と 7 日です。残差は「実測 − 予測」で，正なら不足を表します。方法を替えた改善量は「直線の絶対誤差 − 森の絶対誤差」です。二行でこの値の符号はどうなるでしょうか。

In [ ]:
scored["linear_residual"] = scored["strength"] - scored["linear"]
scored["forest_residual"] = scored["strength"] - scored["forest"]
scored["baseline_error"] = (scored["strength"] - scored["baseline"]).abs()
scored["linear_error"] = scored["linear_residual"].abs()
scored["forest_error"] = scored["forest_residual"].abs()
scored["improvement"] = scored["linear_error"] - scored["forest_error"]

columns_to_read = [
    "record_id", "age", "strength", "linear", "forest",
    "linear_error", "forest_error", "improvement",
]
scored.loc[scored["record_id"].isin([85, 108]), columns_to_read].round(3)

### 平均差と一つずつの改善

二つの MAE を引く場合と，同じ行の改善量を求めて平均する場合の数字を照合します。図の横は改善量で並べた順番です。棒の数と高さから，何行が近づいたかと，全体で誤差がどれだけ減ったかを読み分けます。

In [ ]:
error_columns = ["baseline_error", "linear_error", "forest_error"]
row_mae = scored[error_columns].mean()
gap_from_means = row_mae["linear_error"] - row_mae["forest_error"]
gap_from_rows = scored["improvement"].mean()
print(row_mae.round(3))
print(f"Difference of MAEs: {gap_from_means:.3f} MPa")
print(f"Mean of paired improvements: {gap_from_rows:.3f} MPa")

In [ ]:
change_counts = pd.Series({
    "closer with forest": (scored["improvement"] > 0).sum(),
    "farther with forest": (scored["improvement"] < 0).sum(),
    "same distance": (scored["improvement"] == 0).sum(),
}, name="records")
change_counts

In [ ]:
import matplotlib.pyplot as plt

sorted_improvement = scored["improvement"].sort_values()
fig, ax = plt.subplots(figsize=(7.5, 4.05), layout="constrained")
ax.bar(range(1, len(scored) + 1), sorted_improvement,
       color=["#28617d" if value < 0 else "#a74725" for value in sorted_improvement],
       width=1, linewidth=0)
ax.axhline(0, color="#4b5863", linewidth=1)
ax.set(xlabel="Records ordered by change in absolute error",
       ylabel="Linear error minus forest error (MPa)", xlim=(0, len(scored) + 1))
ax.set_xticks([1, 50, 100, 150, len(scored)])
ax.grid(axis="y", color="#dfe6eb", zorder=0)
ax.set_axisbelow(True)
plt.show()

### 行を数える，配合を数える

架空の配合 A には一行，B には三行あります。それぞれの誤差を一行ずつ四行で平均する場合と，A・B の平均をそれぞれ求めたあと二配合で平均する場合を計算します。二通りでどちらの方法の誤差が小さいかを，実行前に予想してください。

| 配合 | X の絶対誤差（MPa） | Y の絶対誤差（MPa） |
| --- | ---: | ---: |
| A | 0 | 6 |
| B | 4 | 1 |
| B | 4 | 1 |
| B | 4 | 1 |

In [ ]:
toy_errors = pd.DataFrame({
    "recipe": ["A", "B", "B", "B"],
    "X": [0, 4, 4, 4],
    "Y": [6, 1, 1, 1],
})
toy_by_recipe = toy_errors.groupby("recipe")[["X", "Y"]].mean()
toy_comparison = pd.DataFrame({
    "four rows": toy_errors[["X", "Y"]].mean(),
    "two recipes": toy_by_recipe.mean(),
})
toy_comparison

### 実表の二つの平均

検証の 208 行を `recipe_id` でまとめ，各配合の中で絶対誤差を平均します。結果は配合ごとに一行です。それをさらに等しく平均した値と，208 行を直接平均した値を並べます。配合 30 と 116 が持つ元の行数の違いを読み，どちらの計算では何を同じ重さで数えるかを説明します。

In [ ]:
by_recipe = scored.groupby("recipe_id").agg(
    records=("record_id", "size"),
    baseline_error=("baseline_error", "mean"),
    linear_error=("linear_error", "mean"),
    forest_error=("forest_error", "mean"),
)
by_recipe.loc[[30, 116]].round(3)

In [ ]:
recipe_mae = by_recipe[error_columns].mean()
comparison = pd.DataFrame({
    "208 records equally weighted": row_mae,
    "86 recipes equally weighted": recipe_mae,
}, index=error_columns)
comparison.round(3)

行が一つの配合と複数ある配合に分け，それぞれの配合平均の改善量を比べます。図の右は，一点が一配合です。横は直線，縦は森の配合内 MAE です。対角線の下の点はどちらの誤差が小さいでしょうか。

In [ ]:
recipe_improvement = by_recipe["linear_error"] - by_recipe["forest_error"]
recipe_size_comparison = by_recipe.assign(
    several_records=by_recipe["records"] > 1,
    improvement=recipe_improvement,
).groupby("several_records").agg(
    recipes=("records", "size"),
    original_records=("records", "sum"),
    mean_improvement=("improvement", "mean"),
)
recipe_size_comparison.round(3)

In [ ]:
print(f"Closer with forest: {(recipe_improvement > 0).sum()}")
print(f"Farther with forest: {(recipe_improvement < 0).sum()}")
print(f"Same distance: {(recipe_improvement == 0).sum()}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8.55, 4.35), layout="constrained")
size_counts = by_recipe["records"].value_counts().sort_index()
axes[0].bar(size_counts.index, size_counts, width=0.62, color="#28617d")
for records, recipes in size_counts.items():
    axes[0].annotate(str(recipes), (records, recipes), xytext=(0, 3),
                     textcoords="offset points", ha="center", color="#364653")
axes[0].set(xlabel="Records per recipe", ylabel="Number of recipes", ylim=(0, 60))
axes[0].set_xticks(size_counts.index)
axes[0].grid(axis="y", color="#dfe6eb")
axes[0].set_axisbelow(True)

maximum = max(by_recipe["linear_error"].max(), by_recipe["forest_error"].max())
limit = 1.09 * maximum
axes[1].plot([0, limit], [0, limit], color="#687581", linestyle="--", linewidth=1.1)
axes[1].scatter(by_recipe["linear_error"], by_recipe["forest_error"],
                color="#28617d", alpha=0.75, s=26)
focus = by_recipe.loc[30]
axes[1].scatter(focus["linear_error"], focus["forest_error"],
                facecolors="none", edgecolors="#a74725", linewidths=1.8, s=87)
axes[1].annotate("Recipe 30", (focus["linear_error"], focus["forest_error"]),
                 xytext=(-7, 12), textcoords="offset points", ha="right", color="#a74725")
axes[1].set(xlabel="Linear MAE within recipe (MPa)", ylabel="Forest MAE within recipe (MPa)",
            xlim=(0, limit), ylim=(0, limit), aspect="equal")
plt.show()

### 配合 30 の四つの材齢

配合 30 について，3 日と 7 日に続き，28 日と 56 日の実測と予測を照合します。どの方法が近かったかに加え，二つとも実測より高かったり低かったりするところも図から読みます。線は同じ配合の指定材齢を順に結んでいます。

In [ ]:
recipe_30 = scored.loc[scored["recipe_id"] == 30].sort_values("age")
recipe_30[columns_to_read].round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(7.4, 4.25), layout="constrained")
for column, label, color, marker, style in [
    ("strength", "Observed", "#246963", "o", "-"),
    ("linear", "Linear regression", "#8c5621", "s", "--"),
    ("forest", "Random forest", "#225c7a", "D", "-."),
]:
    ax.plot(recipe_30["age"], recipe_30[column], linestyle=style, marker=marker,
            color=color, label=label, markersize=6, linewidth=1.65)
ax.set(xlabel="Age (days)", ylabel="Compressive strength (MPa)", xlim=(0, 60), ylim=(15, 88))
ax.set_xticks(recipe_30["age"])
ax.grid(color="#dfe6eb")
ax.legend(loc="best", frameon=False)
plt.show()

### 次の確認を指定する

講義では，全行の MAE と配合を等しく平均した値の両方で改善が残った森を，次の確認へ進めます。比較するのは，この森と学習平均です。学習の 631 行と八つの入力は同じにし，次回は保留の 191 行を評価します。全行の MAE を主に使い，配合を等しく平均した値と，1〜7 日・8〜28 日・29 日以上の件数と MAE を追加します。保留の範囲別 MAE を検証と同じ範囲の行数比で平均した値も，構成の違いを読む手掛かりにします。ここでは保留の結果を計算しません。

## 実習　自分の比較と次の確認（約 20 分）

### 1. 自分の問いと出力を読み直す（4 分）

第 27 章で保存したノートを開きます。使う範囲と，基本の出力で一つとして数えたものを確認します。コンクリートで予測する人は `"prediction"`，二材齢の強度を記述する人は `"description"` を選びます。第 27 章で選んだ材齢や範囲を使います。変更する場合は理由を書きます。ほかの題材で進める人は `"own"` を選び，自分の基本の図表を使います。

**問いと，二案でそろえる対象**　…

In [ ]:
practice_route = None  # "prediction", "description", "own" のいずれか
if practice_route is None:
    print("自分の問いに合う実習を選んでください。")
elif practice_route not in {"prediction", "description", "own"}:
    raise ValueError("prediction, description, own のいずれかを指定してください。")

### 2. 同じ条件で二案を比べる（6 分）

**予測を選んだ場合**は，検証側の同じ材齢範囲で，直線と森の誤差を比べます。`practice_window` に `"1to7"`，`"8to28"`，`"29plus"` のいずれかを指定します。次に，`practice_rows` に入っている `None` を，`scored` の `age` が `first_day` 以上 `last_day` 以下の行を取り出す式に替えます。例えば，`between()` で範囲の両端を含む条件を書けます。二方法の誤差を同じ行で読むことを確かめてください。

In [ ]:
practice_window = None

if practice_route == "prediction":
    windows = {"1to7": (1, 7), "8to28": (8, 28), "29plus": (29, float("inf"))}
    if practice_window is None:
        print("材齢範囲を一つ選んでください。")
    elif practice_window not in windows:
        raise ValueError("1to7, 8to28, 29plus のいずれかを指定してください。")
    else:
        first_day, last_day = windows[practice_window]
        practice_rows = None  # 選んだ材齢範囲の検証行を取り出す式に替える
        if practice_rows is None:
            print("practice_rows に検証行を取り出す式を書いてください。")
        else:
            expected_index = scored.index[scored["age"].between(first_day, last_day)]
            if not isinstance(practice_rows, pd.DataFrame) or not practice_rows.index.sort_values().equals(expected_index.sort_values()):
                raise ValueError("選んだ材齢範囲と抽出した行が一致していません。scored の age に対する条件を確認してください。")
            practice_score = practice_rows[["linear_error", "forest_error"]].mean()
            print(f"Selected range: {practice_window}")
            print(f"Records: {len(practice_rows)}, recipes: {practice_rows['recipe_id'].nunique()}")
            display(practice_score.rename("MAE (MPa)").round(3))

**記述を選んだ場合**は，学習用の収録記録から，3・7・28・56 日のうち二つの材齢を比べます。`practice_ages` へ二つの数をリストで指定します。次に `rows_at_ages` に入っている `None` を，`train` から `earlier_age` と `later_age` の行を選ぶ式に替えます。第 27 章と同じく `isin()` を使えます。下の集計は，先に両方の材齢がある配合にそろえ，その同じ配合で原行の平均と配合を等しく数えた平均を並べます。二つの差を読むとき，何を等しく数えたかを見てください。

In [ ]:
practice_ages = None

if practice_route == "description":
    if practice_ages is None:
        print("比べる二つの材齢を指定してください。")
    elif (
        not isinstance(practice_ages, (list, tuple))
        or len(practice_ages) != 2
        or practice_ages[0] == practice_ages[1]
        or any(age not in [3, 7, 28, 56] for age in practice_ages)
    ):
        raise ValueError("3・7・28・56 日のうち異なる二つを指定してください。")
    else:
        earlier_age, later_age = sorted(practice_ages)
        rows_at_ages = None  # 学習表から二材齢の行を選ぶ式に替える
        if rows_at_ages is None:
            print("rows_at_ages に二材齢の行を選ぶ式を書いてください。")
        else:
            expected_index = train.index[train["age"].isin([earlier_age, later_age])]
            if not isinstance(rows_at_ages, pd.DataFrame) or not rows_at_ages.index.sort_values().equals(expected_index.sort_values()):
                raise ValueError("選んだ二つの材齢と抽出した行が一致していません。train の age に対する条件を確認してください。")
            per_age = rows_at_ages.groupby(["recipe_id", "age"], as_index=False)["strength"].mean()
            paired_ages = per_age.pivot(index="recipe_id", columns="age", values="strength")
            paired_ages = paired_ages.dropna(subset=[earlier_age, later_age])
            matched_rows = rows_at_ages.loc[rows_at_ages["recipe_id"].isin(paired_ages.index)]
            by_age = matched_rows.groupby("age")["strength"].agg(rows="size", row_mean="mean")
            by_age["recipe_mean"] = paired_ages[[earlier_age, later_age]].mean()
            age_difference = by_age.loc[later_age, ["row_mean", "recipe_mean"]] - by_age.loc[
                earlier_age, ["row_mean", "recipe_mean"]
            ]
            print(f"Recipes with both ages: {len(paired_ages)}")
            display(by_age.round(3))
            display(age_difference.rename("Later minus earlier (MPa)").round(3))

**ほかの題材を使う場合**は，自分の基本の図表で，比べたい二つの集まりや方法を同じ範囲にそろえます。例えば一つの年の記録を記述するなら，確かめたい地域・単位を共通にして，どの集計で特徴が見えるかを考えます。分類の予測なら，検証側の同じ粒と同じ閾値で比べる，などです。比べた二つの数字と分母を，自分の出力で確認します。

**出力から読み取った差と，その分母**　…

### 3. 採る方法と次の確認を決める（7 分）

自分の問いに使う方法または集計を一つ選び，表の数値と，何を一つとして数えたかを根拠に理由を書きます。次に行う確認を一つ決め，その結果はここではまだ計算せずに，予想と説明を改める条件を先に書きます。

| 決めること | 自分の計画 |
| --- | --- |
| 主に使う方法・集計と，その理由 | … |
| 次に何を替え，何をそろえて確かめるか | … |
| どんな結果を予想するか | … |
| 違った場合に，どの説明を見直すか | … |

### 4. 次回に使える記録を残す（3 分）

必要な列と，どの記録を選ぶかが書けたかを確認します。自分のノートへ，今回の表，主に使う方法や集計，次の確認と予想を追記します。配布ノートから始めた場合は，選んだ範囲と計画を書いたものを自分用として保存し，次回もそのファイルを使います。出した表は，入力を残した状態でカーネルを再起動し，上から実行して取り直せます。

### データの出典

データは I-Cheng Yeh の Concrete Compressive Strength（UCI Machine Learning Repository，CC BY 4.0）を使います。教材で元の行番号・配合番号・分割の区分を追加しました。 [UCI の配布ページと利用条件](https://archive.ics.uci.edu/dataset/165/concrete+compressive+strength)